In [75]:
import json
import pandas as pd
import numpy as np
import plotly.express as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [76]:
def parse_lines(lines):
    data = []
    for  line in lines:
        x = json.loads(line)
        data.append(x)

    data = pd.DataFrame.from_dict(data, orient="columns")
    return data

In [77]:
with open("./foo.txt") as f:
    lines = f.readlines()

data = parse_lines(lines)
data = data.set_index(["i", "j", "k", "l"])
data["wctpf"] = data["wctpf"] / 1000;
data["wctpf_label"] = data["wctpf"].map({0: "Fully Mitigated", 25: "Mitigated", 1000000: "Unmitigated"})
data["tpf"] = data["dt"] / data["df"]
data["avg_tpf"] = data["acc_t"] / data["acc_f"]
data["frozen"] = data["irq"].apply(lambda x: True if "Freeze" in x else False)
data

timestamp   fuel   wctpf     dt     df      acc_t    acc_f  \
i j k  l                                                                 
0 0 0  0      5738152  10000  1000.0      0      0          0        0   
       1      5832941  10000  1000.0  93597  10000      93597    10000   
       2      5910868  10000  1000.0  77627  10000     171224    20000   
       3      5987964  10000  1000.0  76906  10000     248130    30000   
       4      6064949  10000  1000.0  76825  10000     324955    40000   
...               ...    ...     ...    ...    ...        ...      ...   
    99 837  138867042  10000  1000.0  87004  10000  133914603  8370000   
       838  138950690  10000  1000.0  83498  10000  133998101  8380000   
       839  139033787  10000  1000.0  82977  10000  134081078  8390000   
       840  139117535  10000  1000.0  83618  10000  134164696  8400000   
       841  139194420  10000  1000.0  76755   9148  134241451  8409148   

                        irq wctpf_label       tpf    avg_tpf  frozen  
i j k  l                                                              
0 0 0  0    {'Unfreeze': 1}         NaN       NaN        NaN   False  
       1    {'Unfreeze': 1}         NaN  9.359700   9.359700   False  
       2    {'Unfreeze': 1}         NaN  7.762700   8.561200   False  
       3    {'Unfreeze': 1}         NaN  7.690600   8.271000   False  
       4    {'Unfreeze': 1}         NaN  7.682500   8.123875   False  
...                     ...         ...       ...        ...     ...  
    99 837  {'Unfreeze': 1}         NaN  8.700400  15.999355   False  
       838  {'Unfreeze': 1}         NaN  8.349800  15.990227   False  
       839  {'Unfreeze': 1}         NaN  8.297700  15.981058   False  
       840  {'Unfreeze': 1}         NaN  8.361800  15.971988   False  
       841  {'Unfreeze': 1}         NaN  8.390359  15.963740   False  

[84200 rows x 12 columns]

In [78]:
x=data.reset_index();
plt.line(x[x["k"]==50], x="l", y="dt", color="wctpf_label")

In [79]:
tpf=data.reset_index()
tpf[["wctpf", "tpf"]].groupby("wctpf").mean()


# plt.box(tpf[tpf["k"] == 40][tpf["tpf"] < 11], x="tpf", color="wctpf", log_x=True)


plt.box(tpf[tpf["k"] == 40][tpf["tpf"] > 100][tpf["wctpf"] != 25], x="tpf", color="wctpf_label", labels={
    "tpf": "Instantanious Execution-Time per Fuel Unit [ns/fuel]",
    "wctpf_label": ""
}, width=800, height=400)



/tmp/ipykernel_388543/560557983.py:8: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_388543/560557983.py:8: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [80]:
avg_tpf=data.groupby(["i", "j", "k"]).tail(1).reset_index()
plt.histogram(avg_tpf, x="avg_tpf", color="wctpf_label", histnorm='percent', nbins=200,
    labels={
        "avg_tpf": "Total Execution-Time per Fuel Unit [ns/fuel]",
        "wctpf_label": ""
        },width=800, height=400
)

In [81]:
x=data.reset_index()
plt.line(x[x["k"] == 0], x="l", y="avg_tpf", color="wctpf_label", labels={"l": "Sample Index (Time)", "avg_tpf": "Avg. Execution-Time per Fuel Unit [ns/fuel]", "wctpf_label": ""}, width=800, height=400)

In [82]:
x = data.reset_index()[["i", "k", "fuel", "timestamp", "acc_t"]]
x = x.groupby(["i", "k"]).agg(["first", "last"])
x["eff"] = x["acc_t"]["last"] / (x["timestamp"]["last"] - x["timestamp"]["first"])
x=x.groupby("i").mean()
plt.line(x=x["fuel"]["first"], y=x["eff"], labels={"x": "Fuel-Chunk Size", "y": "Efficiency"})

In [83]:
x=data.reset_index()
plt.line(x[x["k"] == 0], x="l", y="frozen", color="wctpf_label", labels={"l": "Sample Index (Time)", "frozen": "Is Mitigation Active", "wctpf_label": ""})